In [48]:
# 将数据读取过来，采用前期 数据过滤流程，只采用 prompt，用 qwen3-235b-a22b-instruct-fp8 模型进行后续蒸馏

import os
import json
from tqdm import tqdm 
import pandas as pd

def yield_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f):
            if line.strip():
                # 每次产出一条 JSON 对象
                yield json.loads(line)

def read_jsonl_recursively(root_folder):
    all_data = []
    # os.walk 会返回 (当前路径, 子文件夹列表, 文件名列表)
    for root, dirs, files in os.walk(root_folder):
        for file in files:
            # 检查后缀是否为 .jsonl
            if file.endswith(".jsonl"):
                one_file_datas = []
                type_ = file.split('.')[0]
                over_2_lun = 0
                # 拼接完整的文件路径
                file_path = os.path.join(root, file)
                print(f"正在读取: {file_path}")
                cur_file_length = 0
                for idx, sample in enumerate(yield_data(file_path)):
                    if len(sample['messages']) > 2 and sample['messages'][0]['role'] != 'system':
                        over_2_lun += 1
                        continue
                    for elem in sample['messages']:
                        if elem['role'] == 'user':
                            cur_prompt = elem['content']
                            break
                    # if len(sample['messages']) != 2:
                    #     over_2_lun += 1
                    #     continue
                    # cur_ = {'idx': idx, 'category':type_.split('_')[0], 'source':f'nvidia-cascade-stage1-{type_}--{idx}', 'prompt': sample['messages'][0]['content']}
                    cur_ = {'idx': idx, 'category':type_.split('_')[0], 'source':f'nvidia-cascade-stage1-{type_}--{idx}', 'prompt': cur_prompt}
                    one_file_datas.append(cur_)
                    cur_file_length = idx
                all_data.append(one_file_datas)
                print(f'{file_path}处理完毕，轮次不是一轮的数据包含{over_2_lun}条，总数据量{cur_file_length}条')
    return all_data

In [27]:
stage_1_dir = '/code/zhaoxudong03/zxd_datas/nvidia-cascade-stage1'

all_data = read_jsonl_recursively(stage_1_dir)

正在读取: /code/zhaoxudong03/zxd_datas/nvidia-cascade-stage1/general.jsonl


1171104it [01:50, 10564.07it/s]


/code/zhaoxudong03/zxd_datas/nvidia-cascade-stage1/general.jsonl处理完毕，轮次不是一轮的数据包含124941条，总数据量1171103条
正在读取: /code/zhaoxudong03/zxd_datas/nvidia-cascade-stage1/science.jsonl


239042it [00:55, 4295.07it/s]


KeyboardInterrupt: 

In [16]:
len(all_data)

5311677

In [17]:
all_data[0]

{'idx': 0,
 'category': 'general',
 'source': 'nvidia-cascade-stage1-general--0',
 'prompt': 'Приглашаем на летний отдых у Балтийского моря в нашем миниотеле „Parkvila“.\n\nTranslate this to English?'}

In [20]:
df_all_datas = pd.DataFrame(all_data)
df_all_datas.head()

df_all_datas_dep = df_all_datas.drop_duplicates('prompt')
print(df_all_datas_dep.shape, df_all_datas.shape)

,idx,category,source,prompt
0,0,general,nvidia-cascade-stage1-general--0,Приглашаем на летний отдых у Балтийского моря ...
1,1,general,nvidia-cascade-stage1-general--1,What are some technical skills that you believ...
2,2,general,nvidia-cascade-stage1-general--2,Answer the following multiple-choice question....
3,3,general,nvidia-cascade-stage1-general--3,"My daughter, who is 7 years old, was diagnosed..."
4,4,general,nvidia-cascade-stage1-general--4,Answer the following multiple-choice question....


In [32]:
for group_, elem in df_all_datas.groupby('prompt'):
    if elem.shape[0] != 1:        
        print(type(group_))
        print(elem.shape)
        # print(df_all_datas[df_all_datas['prompt']] == group_)
        print(elem)
        break

<class 'str'>
(8, 4)
            idx category                                source  \
1377856   36511     code   nvidia-cascade-stage1-code_1--36511   
1483013  141668     code  nvidia-cascade-stage1-code_1--141668   
1633956  292611     code  nvidia-cascade-stage1-code_1--292611   
1743556   76813     code   nvidia-cascade-stage1-code_2--76813   
2087695   95554     code   nvidia-cascade-stage1-code_3--95554   
2235375  243234     code  nvidia-cascade-stage1-code_3--243234   
2529169  211630     code  nvidia-cascade-stage1-code_4--211630   
2564339  246800     code  nvidia-cascade-stage1-code_4--246800   

                                                    prompt  
1377856  ![](http://www.grindtv.com/wp-content/uploads/...  
1483013  ![](http://www.grindtv.com/wp-content/uploads/...  
1633956  ![](http://www.grindtv.com/wp-content/uploads/...  
1743556  ![](http://www.grindtv.com/wp-content/uploads/...  
2087695  ![](http://www.grindtv.com/wp-content/uploads/...  
2235375  ![](http:

In [35]:
code_1 = '/code/zhaoxudong03/zxd_datas/nvidia-cascade-stage1/code/code_1.jsonl'
for idx, elem in enumerate(yield_data(code_1)):
    if idx in [36511, 141668, 292611]:
        print(elem)

36482it [00:13, 3021.75it/s]

{'category': 'code', 'source': 'TACO', 'messages': [{'role': 'user', 'content': '![](http://www.grindtv.com/wp-content/uploads/2015/08/drone.jpg)\n\nThe other day I saw an amazing video where a guy hacked some wifi controlled lightbulbs by flying a drone past them. Brilliant.\n\nIn this kata we will recreate that stunt... sort of.\n\nYou will be given two strings: `lamps` and `drone`. `lamps` represents a row of lamps, currently off, each represented by `x`. When these lamps are on, they should be represented by `o`.\n\nThe `drone` string represents the position of the drone `T` (any better suggestion for character??) and its flight path up until this point `=`. The drone always flies left to right, and always begins at the start of the row of lamps. Anywhere the drone has flown, including its current position, will result in the lamp at that position switching on.\n\nReturn the resulting `lamps` string. See example tests for more clarity.\n\nSolve the problem starting with the provide

140967it [00:52, 2983.28it/s]

{'category': 'code', 'source': 'TACO', 'messages': [{'role': 'user', 'content': '![](http://www.grindtv.com/wp-content/uploads/2015/08/drone.jpg)\n\nThe other day I saw an amazing video where a guy hacked some wifi controlled lightbulbs by flying a drone past them. Brilliant.\n\nIn this kata we will recreate that stunt... sort of.\n\nYou will be given two strings: `lamps` and `drone`. `lamps` represents a row of lamps, currently off, each represented by `x`. When these lamps are on, they should be represented by `o`.\n\nThe `drone` string represents the position of the drone `T` (any better suggestion for character??) and its flight path up until this point `=`. The drone always flies left to right, and always begins at the start of the row of lamps. Anywhere the drone has flown, including its current position, will result in the lamp at that position switching on.\n\nReturn the resulting `lamps` string. See example tests for more clarity.\n\nSolve the problem starting with the provide

292717it [01:50, 2492.14it/s]

{'category': 'code', 'source': 'TACO', 'messages': [{'role': 'user', 'content': '![](http://www.grindtv.com/wp-content/uploads/2015/08/drone.jpg)\n\nThe other day I saw an amazing video where a guy hacked some wifi controlled lightbulbs by flying a drone past them. Brilliant.\n\nIn this kata we will recreate that stunt... sort of.\n\nYou will be given two strings: `lamps` and `drone`. `lamps` represents a row of lamps, currently off, each represented by `x`. When these lamps are on, they should be represented by `o`.\n\nThe `drone` string represents the position of the drone `T` (any better suggestion for character??) and its flight path up until this point `=`. The drone always flies left to right, and always begins at the start of the row of lamps. Anywhere the drone has flown, including its current position, will result in the lamp at that position switching on.\n\nReturn the resulting `lamps` string. See example tests for more clarity.\n\nSolve the problem starting with the provide

325398it [02:03, 2644.87it/s]


In [38]:
for g_e, e in df_all_datas_dep.groupby('category'):
    print(g_e, e.shape)

code (159304, 4)
general (768382, 4)
math (348725, 4)
science (67382, 4)


In [39]:
all_datas_dep = df_all_datas_dep.to_dict(orient='records')
all_datas_dep[0]

{'idx': 0,
 'category': 'general',
 'source': 'nvidia-cascade-stage1-general--0',
 'prompt': 'Приглашаем на летний отдых у Балтийского моря в нашем миниотеле „Parkvila“.\n\nTranslate this to English?'}

In [40]:
len(all_datas_dep)

1343793

In [41]:
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/data/nvidia-sft-stage1-134w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(all_datas_dep):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 1343793/1343793 [01:47<00:00, 12480.11it/s]


In [44]:
ori_path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/strict_dep.jsonl'
ori_datas = []
for elem in yield_data(ori_path):
    ori_datas.append(elem)

6721226it [01:10, 95567.92it/s] 


In [45]:
ori_datas[0]

{'prompt': 'def reverse_string(s: str) -> str:\n    """ \n    Reverses the characters in a given string s.\n    \n    >>> reverse_string("hello") == "olleh"\n    >>> reverse_string("") == ""\n    >>> reverse_string("a") == "a"\n    >>> reverse_string("madam") == "madam"\n    >>> reverse_string("hello world") == "dlrow olleh"\n    >>> reverse_string("123! abc") == "cba !321"\n    """',
 'source': 'AM-DeepSeek-R1-0528-Distilled-code-0'}

In [46]:
for elem in all_datas_dep:
    cur_ = {'source': elem['source'], 'prompt': elem['prompt']}
    ori_datas.append(cur_)

In [ ]:
len(ori_datas)

In [50]:
stage_2_dir = '/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2'

all_data_stage2 = read_jsonl_recursively(stage_2_dir)

正在读取: /code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/instruction-following.jsonl


146385it [00:10, 13649.25it/s]


/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/instruction-following.jsonl处理完毕，轮次不是一轮的数据包含0条，总数据量146384条
正在读取: /code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/science.jsonl


311316it [02:02, 2535.88it/s]


/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/science.jsonl处理完毕，轮次不是一轮的数据包含0条，总数据量311315条
正在读取: /code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/swe_localization.jsonl


...

/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/math/math_6.jsonl处理完毕，轮次不是一轮的数据包含0条，总数据量234639条
正在读取: /code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/math/math_7.jsonl


234640it [02:07, 1846.13it/s]


/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/math/math_7.jsonl处理完毕，轮次不是一轮的数据包含0条，总数据量234639条
正在读取: /code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/math/math_8.jsonl


234640it [02:07, 1840.45it/s]

/code/zhaoxudong03/zxd_datas/sft_datas_v2/Nemotron-Cascade-SFT-Stage-2/math/math_8.jsonl处理完毕，轮次不是一轮的数据包含0条，总数据量234639条


In [52]:
len(all_data_stage2)
all_data_stage2_cat = sum(all_data_stage2, [])
len(all_data_stage2_cat)

7160789

In [55]:
df_all_datas_stage2 = pd.DataFrame(all_data_stage2_cat)
df_all_datas_stage2.head()
print(df_all_datas_stage2.shape)
# 删除掉 tool 为category 的数据，这类型数据单独处理训练
df_all_datas_stage2 = df_all_datas_stage2[df_all_datas_stage2['category'] != 'tool'] 
print(df_all_datas_stage2.shape)

(7160789, 4)
(6851992, 4)


In [56]:
df_all_datas_stage2_dep = df_all_datas_stage2.drop_duplicates('prompt')
print(df_all_datas_stage2_dep.shape, df_all_datas_stage2.shape)

(1922829, 4) (6851992, 4)


In [57]:
all_datas_dep_stage2 = df_all_datas_stage2_dep.to_dict(orient='records')

In [58]:
all_datas_dep_stage2[0]

{'idx': 0,
 'category': 'instruction-following',
 'source': 'nvidia-cascade-stage1-instruction-following--0',
 'prompt': 'Please provide a JSON representation of the Patchwork group\'s recent exhibition details. The JSON should include the following fields: "exhibition_title", "start_date", "end_date", "number_of_artworks", and "location". For the "location" field, choose one from the options: "Main Gallery", "East Wing", "Outdoor Pavilion". Also, confirm if the exhibition includes interactive elements by stating: "yes, 100%", "No, no way", or "not sure".'}

In [60]:
for g_e, e in df_all_datas_stage2_dep.groupby('category'):
    print(g_e, e.shape)

code (78852, 4)
general (1163685, 4)
instruction-following (29918, 4)
math (311505, 4)
science (192155, 4)
swe (146714, 4)


In [66]:
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/data/nvidia-sft-stage2-192w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(all_datas_dep_stage2):
        cur_ = {'idx': elem['idx'], 'category': elem['category'], 'source': elem['source'].replace('stage1', 'stage2'), 'prompt': elem['prompt']}
        f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')

100%|██████████| 1922829/1922829 [04:40<00:00, 6842.91it/s] 


In [63]:
all_datas_dep_stage2[0]
for elem in all_datas_dep_stage2:
    cur_ = {'source': elem['source'].replace('stage1', 'stage2'), 'prompt': elem['prompt']}
    ori_datas.append(cur_)

In [67]:
len(ori_datas), ori_datas[0]

(9987848,
 {'prompt': 'def reverse_string(s: str) -> str:\n    """ \n    Reverses the characters in a given string s.\n    \n    >>> reverse_string("hello") == "olleh"\n    >>> reverse_string("") == ""\n    >>> reverse_string("a") == "a"\n    >>> reverse_string("madam") == "madam"\n    >>> reverse_string("hello world") == "dlrow olleh"\n    >>> reverse_string("123! abc") == "cba !321"\n    """',
  'source': 'AM-DeepSeek-R1-0528-Distilled-code-0'})

In [65]:
all_datas_dep_stage2[0]

{'idx': 0,
 'category': 'instruction-following',
 'source': 'nvidia-cascade-stage1-instruction-following--0',
 'prompt': 'Please provide a JSON representation of the Patchwork group\'s recent exhibition details. The JSON should include the following fields: "exhibition_title", "start_date", "end_date", "number_of_artworks", and "location". For the "location" field, choose one from the options: "Main Gallery", "East Wing", "Outdoor Pavilion". Also, confirm if the exhibition includes interactive elements by stating: "yes, 100%", "No, no way", or "not sure".'}

In [68]:
df_ori_datas_cat_stage1_2 = pd.DataFrame(ori_datas)
df_ori_datas_cat_stage1_2_dep = df_ori_datas_cat_stage1_2.drop_duplicates('prompt')
print(df_ori_datas_cat_stage1_2.shape, df_ori_datas_cat_stage1_2_dep.shape)  # 两个 stage 增加了 200w 数据左右

(9987848, 2) (8767864, 2)


In [70]:
df_ori_datas_cat_stage1_2_dep_lst = df_ori_datas_cat_stage1_2_dep.to_dict(orient='records')

In [71]:
len(df_ori_datas_cat_stage1_2_dep_lst), df_ori_datas_cat_stage1_2_dep_lst[0]

(8767864,
 {'prompt': 'def reverse_string(s: str) -> str:\n    """ \n    Reverses the characters in a given string s.\n    \n    >>> reverse_string("hello") == "olleh"\n    >>> reverse_string("") == ""\n    >>> reverse_string("a") == "a"\n    >>> reverse_string("madam") == "madam"\n    >>> reverse_string("hello world") == "dlrow olleh"\n    >>> reverse_string("123! abc") == "cba !321"\n    """',
  'source': 'AM-DeepSeek-R1-0528-Distilled-code-0'})

In [72]:
with open('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/data/strict_dep_cat_nvidia_two_stages_876w.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(df_ori_datas_cat_stage1_2_dep_lst):
        f_save.write(json.dumps(elem, ensure_ascii=False)+'\n')

100%|██████████| 8767864/8767864 [12:09<00:00, 12021.10it/s]


In [76]:
total_202w_only_156w_datas = []
with open('/code/zhaoxudong03/data_pipelines/202w_totals/total_202w_only_156w_prompt.jsonl', 'w', encoding='utf8') as f_save:
    for elem in tqdm(yield_data('/code/zhaoxudong03/data_pipelines/202w_totals/total_202w_only_156w_embedding.jsonl')):
        cur_ = {'source': elem['source'], 'prompt': elem['instruction']+elem['input']}
        total_202w_only_156w_datas.append(cur_)
        f_save.write(json.dumps(cur_, ensure_ascii=False)+'\n')
    

0it [00:00, ?it/s]
6it [00:00, 37.09it/s]
19it [00:01, 15.60it/s][A
69it [00:04, 14.67it/s]
248it [00:06, 50.74it/s][A
560it [00:06, 145.56it/s][A
642it [00:06, 172.82it/s]
1082it [00:06, 413.22it/s][A
1278it [00:07, 336.12it/s]
1419it [00:08, 261.95it/s]
1740it [00:08, 426.19it/s]
2175it [00:08, 715.33it/s]
2426it [00:09, 447.56it/s]
2607it [00:10, 325.56it/s]
3042it [00:10, 538.53it/s]
3418it [00:11, 484.11it/s]
3851it [00:11, 713.84it/s]
4268it [00:11, 985.48it/s]
4567it [00:12, 596.53it/s]
5011it [00:13, 856.08it/s]
5296it [00:13, 603.19it/s]
5744it [00:14, 868.49it/s]
6144it [00:15, 636.37it/s]
6586it [00:15, 885.73it/s]
7011it [00:15, 1175.12it/s][A
7333it [00:16, 621.52it/s] 
7662it [00:16, 794.85it/s] 
7969it [00:17, 589.61it/s]
8403it [00:17, 842.56it/s]
8780it [00:17, 1099.96it/s][A
9079it [00:18, 612.92it/s] 
9505it [00:18, 865.82it/s] 
9785it [00:20, 517.23it/s]
10154it [00:20, 708.14it/s][A
10595it [00:20, 1001.04it/s][A
10904it [00:21, 687.76it/s] 
11345it [00:21, 970.61i

KeyboardInterrupt: 

In [ ]:
new_datas_path = '/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/data/strict_dep_cat_nvidia_two_stages_876w_01_strict_dep_for_embedding.jsonl'

total_202w_only_156w_cat_new_datas = []
for elem in tqdm(yield_data(new_datas_path)):
    total_202w_only_156w_cat_new_datas.append(elem)